# colab_07b — Re-parse Bhaduri 2020 with raw counts (drop-in 100k replacement)

## Motivation

Session 22 closed with three failed Harmony integrations (colab_08, 08b, 08c). Escalating to scVI — a deep generative integration method that conditions on batch label and uses a negative binomial likelihood. **scVI requires raw integer counts.**

The existing `bhaduri_2020_100k.h5ad` has log-normalized data in both `.X` and `.raw.X`. The likely cause: in colab_01, `adata.raw = adata` was called and then `sc.pp.normalize_total` modified `.X` in-place; due to scanpy's memory aliasing, the modification propagated through to `.raw.X` as well. The original `bhaduri_2020_raw.h5ad` (colab_00 output) is no longer on Drive.

Fortunately the source GEO matrix `GSE132672_allorganoids_withnew_matrix.txt.gz` is still on Drive. This notebook:

1. Re-parses the GEO matrix into a fresh raw-count AnnData (mirrors colab_00's parser exactly).
2. Looks up the same 100,000 barcodes that `bhaduri_2020_100k.h5ad` already contains and subsets the freshly-parsed full matrix to those exact cells.
3. Copies obs metadata (protocol, age_week, sample, leiden) from the existing 100k file.
4. Saves `bhaduri_2020_100k_counts.h5ad` — a drop-in replacement where `.X` is genuine integer counts.

Bhaduri 2021 already has raw counts in `.X` of its 100k file (verified in colab_08 section 2) — no work needed for that side. Recommended Colab runtime: high-RAM. The full 242k×16,774 sparse matrix fits comfortably (~4 GB peak), but the parse step accumulates Python-side `array.array` buffers before the sparse build.

## Section 0 — Setup

### 0a — Install, mount, define paths

Three Drive locations matter:
- **Input**: `data/raw/bhaduri_2020/GSE132672_allorganoids_withnew_matrix.txt.gz` (~4 GB on disk, gzipped TSV genes×cells)
- **Intermediate**: `data/processed/bhaduri_2020/bhaduri_2020_raw_counts.h5ad` — fresh full re-parse, raw integer counts. Replaces the lost `bhaduri_2020_raw.h5ad`.
- **Final output**: `data/processed/bhaduri_2020/bhaduri_2020_100k_counts.h5ad` — the same 100k cells as `bhaduri_2020_100k.h5ad`, but `.X` is raw counts.

Also loads the existing `bhaduri_2020_100k.h5ad` location for barcode lookup in section 2.

In [1]:
!pip install -q scanpy

from google.colab import drive
drive.mount('/content/drive')

import os
import io
import gzip
import array
import numpy as np
import scipy.sparse as sp
import anndata
import scanpy as sc

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'
PATHS = {
    'geo_matrix':    os.path.join(DRIVE_ROOT, 'data/raw/bhaduri_2020/GSE132672_allorganoids_withnew_matrix.txt.gz'),
    'raw_counts':    os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_raw_counts.h5ad'),
    'existing_100k': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad'),
    'final_100k':    os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_100k_counts.h5ad'),
}

print('scanpy', sc.__version__, '| anndata', anndata.__version__)
print()
for k, v in PATHS.items():
    marker = 'OK     ' if os.path.exists(v) else 'MISSING'
    print(f'  [{marker}] {k}: {v}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Mounted at /content/drive


/tmp/ipykernel_4891/2548682819.py:23: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print('scanpy', sc.__version__, '| anndata', anndata.__version__)
/tmp/ipykernel_4891/2548682819.py:23: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print('scanpy', sc.__version__, '| anndata', anndata.__version__)


scanpy 1.12.1 | anndata 0.12.11

  [OK     ] geo_matrix: /content/drive/MyDrive/brain-organoid-trajectories/data/raw/bhaduri_2020/GSE132672_allorganoids_withnew_matrix.txt.gz
  [MISSING] raw_counts: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_raw_counts.h5ad
  [OK     ] existing_100k: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad
  [MISSING] final_100k: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_100k_counts.h5ad


## Section 1 — Re-parse the GEO matrix into a raw-count AnnData

### 1a — Streaming sparse parser (mirrors colab_00 exactly)

The matrix is genes × cells in gzipped TSV — too large for pandas (ca. 40 GB RAM if dense). Custom streaming parser uses `array.array` for C-primitive storage (ca. 7× RAM reduction vs Python lists), reads one gene row at a time, stores only non-zero values, then builds a sparse CSR matrix and transposes to cells×genes.

This is the **exact same parser as colab_00**. Determinism is critical here: section 2's barcode lookup requires that this parse produces the same 242,349 cell barcodes (and 16,774 gene names in the same order) as the original `bhaduri_2020_raw.h5ad` that we lost. Same input file + same parser code = same output.

In [2]:
print('Reading Bhaduri 2020 — fast sparse parser...')

data        = array.array('f')   # float32 values
col_indices = array.array('i')   # int32 column indices
indptr      = [0]
gene_names  = []
cell_names  = None

with io.TextIOWrapper(io.BufferedReader(gzip.open(PATHS['geo_matrix'], 'rb'), buffer_size=64 * 1024 * 1024)) as f:
    cell_names = f.readline().rstrip('\n').split('\t')  # first row = cell barcodes

    for i, line in enumerate(f):
        line = line.rstrip('\n')
        tab = line.index('\t')
        gene_names.append(line[:tab])

        vals = np.fromstring(line[tab + 1:], dtype=np.float32, sep='\t')
        nz = np.flatnonzero(vals)
        data.extend(vals[nz].tolist())
        col_indices.extend(nz.astype(np.int32).tolist())
        indptr.append(indptr[-1] + len(nz))

        if (i + 1) % 2000 == 0:
            print(f'  {i + 1:,} genes processed...', flush=True)

print('Building sparse matrix...')
X = sp.csr_matrix(
    (np.frombuffer(data, dtype=np.float32),
     np.frombuffer(col_indices, dtype=np.int32),
     np.array(indptr, dtype=np.int32)),
    shape=(len(gene_names), len(cell_names))
).T.tocsr()

adata_full = anndata.AnnData(X)
adata_full.obs_names = cell_names
adata_full.var_names = gene_names

print(f'\nDone. Shape: {adata_full.shape[0]:,} cells x {adata_full.shape[1]:,} genes')
print(f'Sparsity: {100 * (1 - X.nnz / (X.shape[0] * X.shape[1])):.1f}% zeros')

Reading Bhaduri 2020 — fast sparse parser...
  2,000 genes processed...
  4,000 genes processed...
  6,000 genes processed...
  8,000 genes processed...
  10,000 genes processed...
  12,000 genes processed...
  14,000 genes processed...
  16,000 genes processed...
Building sparse matrix...

Done. Shape: 242,350 cells x 16,774 genes
Sparsity: 92.2% zeros


In [4]:
import numpy as np
row_sums = np.asarray(adata_full.X.sum(axis=1)).ravel()
print(f'Row sums (per-cell totals):')
print(f'  min:    {row_sums.min():.1f}')
print(f'  median: {np.median(row_sums):.1f}')
print(f'  max:    {row_sums.max():.1f}')
print(f'  std:    {row_sums.std():.1f}')
print(f'\nFirst 10: {row_sums[:10]}')

Row sums (per-cell totals):
  min:    0.0
  median: 63516.7
  max:    8058962.0
  std:    152929.0

First 10: [305542.6    53458.58   31004.234  42572.93   65508.453  70000.13
  41466.652  53792.43   96541.31   49487.695]


### 1b — Confirm integer-valued counts and save full raw h5ad

Sanity check: `.X.data` should contain non-negative integers stored as float32 (the parser casts to float during accumulation). If max < 10 or values are non-integer, the parsed file is something other than raw counts and we stop here.

Saving the full re-parse to `bhaduri_2020_raw_counts.h5ad` is cheap insurance: any future analysis that needs counts (alternative subsamples, NB-based DE, count-aware tools) starts from this file instead of re-parsing the 4 GB GEO matrix.

In [3]:
print(f'Shape: {adata_full.shape}')
print(f'.X dtype: {adata_full.X.dtype}')
print(f'.X format: {adata_full.X.format}')
print(f'.X.data first 10 nonzero values: {adata_full.X.data[:10]}')
print(f'.X.data max: {adata_full.X.data.max()}')
print(f'.X.data min nonzero: {adata_full.X.data.min()}')

is_integer_valued = np.allclose(adata_full.X.data, np.round(adata_full.X.data))
print(f'\nValues are integer-valued: {is_integer_valued}')
if not is_integer_valued:
    raise ValueError('Parsed values are not integer — input file is not raw counts. Stop.')

os.makedirs(os.path.dirname(PATHS['raw_counts']), exist_ok=True)
adata_full.write_h5ad(PATHS['raw_counts'])
print(f'\nSaved: {PATHS["raw_counts"]}')
print(f'  size: {os.path.getsize(PATHS["raw_counts"]) / 1e6:.1f} MB')

Shape: (242350, 16774)
.X dtype: float32
.X format: csr
.X.data first 10 nonzero values: [ 1.7689409 19.036036   8.452     42.365913  16.44358   57.10811
  7.725777   8.731405   7.2487135  9.827907 ]
.X.data max: 711374.625
.X.data min nonzero: 0.211735337972641

Values are integer-valued: False


ValueError: Parsed values are not integer — input file is not raw counts. Stop.

## Section 2 — Look up the existing 100k subsample by barcode

### 2a — Load existing 100k file, capture target barcodes and obs metadata

The existing `bhaduri_2020_100k.h5ad` (built by colab_07) is the source of truth for *which* 100k cells were chosen. Load it and capture: `obs_names` (the 100k target barcodes), full `obs` dataframe (protocol, age_week, sample, leiden), and `var` dataframe (gene metadata: mt flag, etc.).

We deliberately do not use this file's `.X` for anything downstream — it has log-normalized data, which is what we are replacing.

In [ ]:
adata_existing = sc.read_h5ad(PATHS['existing_100k'])
print(f'Existing 100k file shape: {adata_existing.shape}')
print(f'obs columns: {list(adata_existing.obs.columns)}')
print(f'var columns: {list(adata_existing.var.columns)}')
print(f'First 3 barcodes: {adata_existing.obs_names[:3].tolist()}')

target_barcodes = adata_existing.obs_names.tolist()
target_obs = adata_existing.obs.copy()
target_var = adata_existing.var.copy()

print(f'\nCaptured: {len(target_barcodes):,} barcodes, {target_obs.shape[1]} obs columns, {target_var.shape[1]} var columns')

### 2b — Subset full raw matrix to the target barcodes (in target order)

Build a barcode → row-index map from `adata_full`, then collect positions in the order of `target_barcodes`. Slicing in target order means the new file's `obs_names` are aligned 1:1 with the existing file's, so we can paste obs metadata directly in section 3 without reindexing.

If any target barcode is not found in the re-parse, the parser was non-deterministic with colab_00 — hard stop, do not proceed.

In [ ]:
full_index = adata_full.obs_names
barcode_to_pos = {bc: i for i, bc in enumerate(full_index)}

missing = [bc for bc in target_barcodes if bc not in barcode_to_pos]
print(f'Target barcodes: {len(target_barcodes):,}')
print(f'Missing in re-parse: {len(missing):,}')
if missing:
    print(f'First 5 missing: {missing[:5]}')
    raise ValueError('Re-parse does not contain all target barcodes — parser non-deterministic. Stop.')

positions = [barcode_to_pos[bc] for bc in target_barcodes]
adata_100k = adata_full[positions].copy()

print(f'\nSubsetted shape: {adata_100k.shape}')
print(f'First 3 obs_names new:      {adata_100k.obs_names[:3].tolist()}')
print(f'First 3 obs_names existing: {adata_existing.obs_names[:3].tolist()}')
print(f'obs_names exactly aligned: {(adata_100k.obs_names == adata_existing.obs_names).all()}')

### 2c — Verify gene-space identity with existing 100k file

Both files should have 16,774 genes in the same order. If gene order differs, scVI will silently produce wrong results because var metadata (mt flag, etc.) refers to genes by position. Assert exact identity; if the gene set matches but order differs, reorder to match the existing file.

In [ ]:
print(f'New gene count:      {adata_100k.shape[1]:,}')
print(f'Existing gene count: {adata_existing.shape[1]:,}')
print(f'First 5 var_names new:      {adata_100k.var_names[:5].tolist()}')
print(f'First 5 var_names existing: {adata_existing.var_names[:5].tolist()}')

genes_match_count = adata_100k.shape[1] == adata_existing.shape[1]
genes_match_set   = set(adata_100k.var_names) == set(adata_existing.var_names)
genes_match_order = genes_match_count and (adata_100k.var_names == adata_existing.var_names).all()

print(f'\nGene count match: {genes_match_count}')
print(f'Gene set match:   {genes_match_set}')
print(f'Gene order match: {genes_match_order}')

if not genes_match_order and genes_match_set:
    print('\nGenes are the same set but different order — reordering to match existing.')
    adata_100k = adata_100k[:, adata_existing.var_names].copy()
    print(f'After reorder, gene order match: {(adata_100k.var_names == adata_existing.var_names).all()}')
elif not genes_match_set:
    raise ValueError('Gene sets differ between re-parse and existing 100k. Stop and investigate.')

## Section 3 — Copy obs and var metadata from existing 100k

### 3a — Attach captured obs and var

Freshly-parsed `adata_100k` has empty obs (only barcodes) and minimal var (only gene names). Replace both with the captured frames from section 2a. Asserts that `obs_names` and `var_names` are aligned before assignment so that no metadata is silently misattributed.

In [ ]:
assert (adata_100k.obs_names == adata_existing.obs_names).all(), 'obs_names misaligned — do not assign'
assert (adata_100k.var_names == adata_existing.var_names).all(), 'var_names misaligned — do not assign'

adata_100k.obs = target_obs.copy()
adata_100k.var = target_var.copy()

print(f'obs columns: {list(adata_100k.obs.columns)}')
print(f'var columns: {list(adata_100k.var.columns)}')

if 'protocol' in adata_100k.obs.columns:
    print(f'\nProtocol distribution:')
    print(adata_100k.obs['protocol'].value_counts())
if 'age_week' in adata_100k.obs.columns:
    print(f'\nAge_week distribution:')
    print(adata_100k.obs['age_week'].value_counts().sort_index())

# Free the full adata_full now that subsetting is complete
del adata_full
import gc; gc.collect()

## Section 4 — Save and verify

### 4a — Write `bhaduri_2020_100k_counts.h5ad` to Drive

Final output. `.X` = raw integer counts (float32). Same 100,000 cells, same 16,774 genes, same obs metadata as `bhaduri_2020_100k.h5ad` — the only difference is `.X` is now actual counts. No `.raw` is set (would just duplicate `.X`).

In [ ]:
os.makedirs(os.path.dirname(PATHS['final_100k']), exist_ok=True)
adata_100k.write_h5ad(PATHS['final_100k'])
print(f'Saved: {PATHS["final_100k"]}')
print(f'  size: {os.path.getsize(PATHS["final_100k"]) / 1e6:.1f} MB')

### 4b — Reload and final verification

Load the freshly-saved file from Drive (round-trip check) and confirm: shape (100,000, 16,774), `.X` dtype float32, integer-valued data, obs columns preserved, barcode set identical to existing 100k. This is the go/no-go gate for colab_08d (scVI).

In [ ]:
adata_check = sc.read_h5ad(PATHS['final_100k'])
print(f'Reloaded shape: {adata_check.shape}')
print(f'.X dtype: {adata_check.X.dtype}')
print(f'.X format: {adata_check.X.format}')
print(f'.X.data first 10 nonzero: {adata_check.X.data[:10]}')
print(f'.X.data max: {adata_check.X.data.max()}')
print(f'.X.data nnz: {adata_check.X.nnz:,}')

is_integer_valued = np.allclose(adata_check.X.data, np.round(adata_check.X.data))
print(f'\nValues integer-valued: {is_integer_valued}')

# Cross-check vs existing
new_set = set(adata_check.obs_names)
existing_set = set(adata_existing.obs_names)
print(f'\nBarcode identity vs existing 100k:')
print(f'  intersection: {len(new_set & existing_set):,}')
print(f'  only in new:      {len(new_set - existing_set):,}')
print(f'  only in existing: {len(existing_set - new_set):,}')

print(f'\nobs columns: {list(adata_check.obs.columns)}')
print(f'var columns: {list(adata_check.var.columns)}')

all_pass = (
    adata_check.shape == (100_000, 16_774)
    and is_integer_valued
    and new_set == existing_set
)
print(f'\nAll checks pass: {all_pass}')
if all_pass:
    print('Ready for colab_08d (scVI).')
else:
    print('At least one check failed — investigate before colab_08d.')